In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))
from src.common.config import PipelineConfig
from src.common.gobierno import aplicar

cfg = PipelineConfig(fecha_corte="2026-11-05")
resultado = aplicar(spark, cfg)
print(f"OK: {resultado['ok']}  |  Con error: {len(resultado['fallidas'])}")
for sentencia, error in resultado["fallidas"]:
    print("-", sentencia, "->", error)

In [0]:
CATALOGO, ESQUEMA = "workspace", "default"
T = lambda n: f"{CATALOGO}.{ESQUEMA}.{n}"

permisos = []
# Todos necesitan "entrar" al catálogo y al esquema
for g in ["grp_ingenieria_datos", "grp_analistas", "grp_auditoria"]:
    permisos += [f"GRANT USE CATALOG ON CATALOG {CATALOGO} TO `{g}`",
                 f"GRANT USE SCHEMA ON SCHEMA {CATALOGO}.{ESQUEMA} TO `{g}`"]

# Ingeniería: opera el pipeline
permisos += [
    f"GRANT CREATE TABLE, CREATE FUNCTION, SELECT, MODIFY, EXECUTE ON SCHEMA {CATALOGO}.{ESQUEMA} TO `grp_ingenieria_datos`",
    f"GRANT READ VOLUME, WRITE VOLUME ON VOLUME {T('volume1')} TO `grp_ingenieria_datos`",
]

# Analistas: SOLO Gold y vistas protegidas (nunca silver_clientes directo)
for t in ["gold_poliza_360", "gold_siniestralidad_mensual", "gold_discrepancias",
          "vw_clientes_protegido", "vw_recaudos_protegido"]:
    permisos.append(f"GRANT SELECT ON TABLE {T(t)} TO `grp_analistas`")
for f in ["fn_mask_ultimos", "fn_mask_correo"]:
    permisos.append(f"GRANT EXECUTE ON FUNCTION {T(f)} TO `grp_analistas`")

# Auditoría: SOLO control, cuarentena y calidad
for t in ["ctl_ejecuciones", "ctl_cargas", "ctl_esquemas", "ctl_calidad_resultados", "gold_calidad_pipeline",
          "quarantine_clientes", "quarantine_polizas", "quarantine_recaudos",
          "quarantine_siniestros", "quarantine_movimientos"]:
    permisos.append(f"GRANT SELECT ON TABLE {T(t)} TO `grp_auditoria`")

ok, errores = 0, []
for p in permisos:
    try:
        spark.sql(p); ok += 1
    except Exception as e:
        errores.append((p, str(e).splitlines()[0][:150]))

print(f"Permisos aplicados: {ok} de {len(permisos)}")
for p, e in errores:
    print("ERROR:", p, "\n   ->", e)

In [0]:
%sql
SELECT cliente_id, numero_documento, correo, telefono, fecha_nacimiento
FROM workspace.default.vw_clientes_protegido LIMIT 5;

In [0]:
%sql
SELECT table_name, column_name, tag_name, tag_value
FROM workspace.information_schema.column_tags
WHERE schema_name = 'default'
ORDER BY table_name, column_name;

In [0]:
%sql
SHOW GRANTS ON TABLE workspace.default.gold_poliza_360;